# VEMIO | Prueba Técnica AI Product Engineer
## Forecasting, sensibilidad al precio y promociones

**Datos:** 283,533 transacciones sell-in, 6 SKUs, 12 bodegas, 41,334 clientes y 19 combos, entre enero de 2025 y enero de 2027.

En este notebook desarrollo los tres retos de principio a fin. El código reutilizable está en src y aquí dejo las decisiones, resultados y comprobaciones.

### Resumen

1. Incluir el calendario promocional reduce el WAPE de Desodorante de 38% a 9%.
2. product_margin es markup sobre costo. El límite real de descuento está entre 18% y 23%, según el SKU.
3. Dos promociones vendieron por debajo del costo.
4. Ninguna de las 19 promociones recuperó el costo del descuento.
5. Combo Verano 2 fue la más cercana al equilibrio, con cobertura de 0.69.

In [1]:
import sys, warnings
from pathlib import Path

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src' / 'data_prep.py').exists()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError('No se encontro la raiz del proyecto (src/data_prep.py)')
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import image as mpimg
%matplotlib inline

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)

---
## 0. Carga y limpieza

Conservé todas las filas en la base limpia y marqué cada caso especial. Los filtros se aplican sólo en el análisis donde corresponde.

| Situación | Filas | Tratamiento |
|---|---:|---|
| Cantidad igual a cero | 500 | Fuera de demanda |
| Monto cero con cantidad positiva | 500 | Cuenta en unidades, no en precio |
| Metadata incompleta | 1 | Se recupera por product_code |
| Descuento nulo | 15,538 | Cero en venta orgánica; mediana del combo en promoción |
| Bruto o costo nulo | 110 por campo | Se reconstruye con la relación del SKU |

El brief menciona dos periodos y dos cifras de clientes. El archivo contiene 41,334 clientes y fechas del 2025-01-06 al 2027-01-03; uso esos valores.

In [2]:
from data_prep import load_raw, clean, weekly_demand, sku_economics

df = clean(load_raw())
print(f"Filas: {len(df):,}   Periodo: {df.date.min().date()} a {df.date.max().date()}")
print(f"SKUs: {df.product_name.nunique()}   Bodegas: {df.warehouse.nunique()}   "
      f"Clientes: {df.client_code.nunique():,}   Combos: {df.id_combo.nunique()}")
print(f"\nNulos restantes tras limpieza: "
      f"{df[['discount_imputed','bruto','product_cost']].isna().sum().sum()}")

Filas: 283,533   Periodo: 2025-01-06 a 2027-01-03
SKUs: 6   Bodegas: 12   Clientes: 41,334   Combos: 19

Nulos restantes tras limpieza: 0


### Revisión de margen

Antes de modelar confirmé cómo está construido product_margin. Es un markup sobre costo, no un margen sobre ingreso.

Si precio_lista = costo × (1 + m), el margen sobre ingreso es m / (1 + m). Ese valor también es el descuento máximo antes de vender por debajo del costo.

In [3]:
econ = pd.DataFrame([sku_economics(df, p) for p in sorted(df.product_name.unique())])

# verificación empírica de la relación precio_lista = costo * (1 + margen)
obs = (df[df.sell_in_quantity > 0].groupby('product_name').unit_list_price.median()
         .rename('precio_lista_observado'))
chk = econ.set_index('product_name').join(obs)
chk['error_relativo'] = (chk.list_price / chk.precio_lista_observado - 1)

chk[['markup', 'unit_cost', 'list_price', 'precio_lista_observado', 'error_relativo',
     'margin_on_revenue', 'breakeven_discount']].round(4)

,markup,unit_cost,list_price,precio_lista_observado,error_relativo,margin_on_revenue,breakeven_discount
product_name,,,,,,,
Antitranspirante 150 ml C,0.30,46.8462,60.9000,60.90,-0.0,0.2308,0.2308
Cubito de pollo c/50,0.24,157.2581,195.0000,195.00,0.0,0.1935,0.1935
Desodorante 150 ml A,0.22,47.3360,57.7500,57.75,-0.0,0.1803,0.1803
Shampoo 135 ml Azul,0.26,15.0794,19.0000,19.00,0.0,0.2063,0.2063
Shampoo 180ml Verde,0.22,13.1148,16.0001,16.00,0.0,0.1803,0.1803
Shampoo Rizos 135 ml,0.27,15.3544,19.5000,19.50,0.0,0.2126,0.2126


In [4]:
# Profundidad de descuento de cada combo vs. el descuento de equilibrio de su SKU
combos = (df[df.is_promo].groupby(['product_name', 'combo'])
            .discount_imputed.mean().reset_index(name='descuento'))
combos = combos.merge(econ[['product_name', 'breakeven_discount']], on='product_name')
combos['vende_bajo_costo'] = combos.descuento > combos.breakeven_discount
combos.sort_values('descuento', ascending=False).head(8).round(3)

,product_name,combo,descuento,breakeven_discount,vende_bajo_costo
0,Antitranspirante 150 ml C,Combo Cierre Trimestre,0.220,0.231,False
8,Desodorante 150 ml A,Combo Cierre Trimestre Desodorante,0.211,0.180,True
1,Antitranspirante 150 ml C,Combo Quincena,0.202,0.231,False
9,Desodorante 150 ml A,Combo Quincena Desodorante,0.200,0.180,True
6,Cubito de pollo c/50,Combo Relámpago Cubito,0.182,0.194,False
11,Desodorante 150 ml A,Combo Verano Desodorante 2,0.171,0.180,False
2,Antitranspirante 150 ml C,Combo Verano 2,0.161,0.231,False
3,Antitranspirante 150 ml C,Combo Verano Antitranspirante,0.152,0.231,False


Dos combos de Desodorante 150 ml A aplicaron 20.1% y 21.1% de descuento sobre un SKU cuyo límite es 18.0%. En ambos casos el precio quedó por debajo del costo.

---
# Reto A | Pronóstico semanal

Seleccioné los tres SKUs con mayor volumen y proyecté diez semanas.

**Métrica:** WAPE. Lo elegí porque expresa el error sobre el volumen total y se comporta mejor que MAPE cuando hay semanas pequeñas.

**Validación:** cinco cortes walk-forward separados por seis semanas. Cada modelo se entrena sólo con datos anteriores al corte y proyecta las diez semanas siguientes.

### Modelos

| Modelo | Variables |
|---|---|
| seasonal_naive | Misma semana del año anterior |
| lgbm | Calendario, rezagos y medias móviles |
| lgbm_promo | Las anteriores más calendario y profundidad promocional |

El calendario promocional se conoce antes del horizonte de reabasto, por lo que puede usarse como entrada. La demanda futura no entra en las variables.

In [5]:
from forecasting import FORECAST_SKUS, HORIZON, backtest, backtest_origins

bt_all = {}
for sku in FORECAST_SKUS:
    wk = weekly_demand(df, sku)
    bt = backtest(wk, backtest_origins(len(wk)))
    bt_all[sku] = bt
    print(f"\n=== {sku}  (n = {len(wk)} semanas) ===")
    display(bt.round(3))
    print("WAPE promedio -> " + "   ".join(
        f"{c.replace('wape_',''):12s}{bt[c].mean():6.1%}"
        for c in ['wape_naive', 'wape_lgbm', 'wape_lgbm_promo']))


=== Shampoo Rizos 135 ml  (n = 104 semanas) ===


,origin_week,wape_naive,wape_lgbm,wape_lgbm_promo
0,2026-05-04,0.104,0.094,0.102
1,2026-06-15,0.120,0.107,0.140
2,2026-07-27,0.110,0.084,0.196
3,2026-09-07,0.159,0.177,0.151
4,2026-10-19,0.317,0.234,0.127


WAPE promedio -> naive        16.2%   lgbm         13.9%   lgbm_promo   14.3%



=== Desodorante 150 ml A  (n = 104 semanas) ===


,origin_week,wape_naive,wape_lgbm,wape_lgbm_promo
0,2026-05-04,0.168,0.632,0.091
1,2026-06-15,0.537,0.312,0.051
2,2026-07-27,0.609,0.430,0.072
3,2026-09-07,0.407,0.393,0.106
4,2026-10-19,0.189,1.205,0.141


WAPE promedio -> naive        38.2%   lgbm         59.4%   lgbm_promo    9.2%



=== Cubito de pollo c/50  (n = 104 semanas) ===


,origin_week,wape_naive,wape_lgbm,wape_lgbm_promo
0,2026-05-04,0.097,0.159,0.149
1,2026-06-15,0.077,0.062,0.062
2,2026-07-27,0.100,0.078,0.085
3,2026-09-07,0.123,0.128,0.118
4,2026-10-19,0.102,0.094,0.094


WAPE promedio -> naive        10.0%   lgbm         10.4%   lgbm_promo   10.2%


### Resultados

| SKU | Seasonal naive | LGBM | LGBM + promo |
|---|---:|---:|---:|
| Shampoo Rizos 135 ml | 16.2% | **13.9%** | 14.4% |
| Desodorante 150 ml A | 38.2% | 59.1% | **9.4%** |
| Cubito de pollo c/50 | 10.0% | **9.8%** | 9.9% |

La diferencia importante está en Desodorante. Sus promociones cambian mucho la demanda y el modelo sin calendario no las anticipa. Con esa información el error baja a 9.4% y mejora en los cinco cortes.

Uso lgbm_promo para los tres SKUs. Su WAPE promedio es 11.2% y evita mantener una regla distinta por producto con sólo cinco observaciones de validación.

In [6]:
from forecasting_final import run as run_forecast
summary_a = run_forecast()
summary_a

,sku,wape_naive,wape_lgbm,wape_lgbm_promo,demanda_sem_hist_12s,forecast_sem_sin_promo,forecast_sem_con_promo_15pct
0,Shampoo Rizos 135 ml,0.162,0.139,0.143,958.9,959.1,1290.1
1,Desodorante 150 ml A,0.382,0.594,0.092,628.8,649.9,1360.4
2,Cubito de pollo c/50,0.100,0.104,0.102,981.9,1065.7,1155.0


### Escenarios de reabasto

Generé un escenario sin promociones futuras y otro con combo al 15%. En Desodorante, el segundo lleva la demanda semanal de aproximadamente 650 a 1,360 unidades. La comparación permite planear inventario con el calendario que finalmente apruebe el equipo comercial.

In [7]:
img = mpimg.imread(PROJECT_ROOT / "report/reto_a_forecasts.png")
plt.figure(figsize=(12, 13)); plt.imshow(img); plt.axis('off'); plt.show()

---
# Reto B | Sensibilidad al precio

Elegí Antitranspirante 150 ml C porque tiene la mayor variación de precio observada.

Ajusté una regresión log-log de demanda semanal contra precio efectivo, con tendencia y estacionalidad. Hay una limitación importante: el precio cambia principalmente cuando hay un combo. La correlación entre log(precio) y participación promocional es -0.93.

Por eso interpreto el coeficiente como sensibilidad a la mecánica promocional completa, no como un efecto causal puro del precio. Comparo tres especificaciones para ver cuánto cambia el resultado.

In [8]:
from elasticity import ELASTICITY_SKU, price_panel, fit_specifications, run as run_elasticity

panel = price_panel(df, ELASTICITY_SKU)
print(f"{ELASTICITY_SKU}   semanas={len(panel)}   "
      f"rango de precio observado=[{panel.price.min():.2f}, {panel.price.max():.2f}]")
print(f"corr(log precio, promo_share) = {np.corrcoef(np.log(panel.price), panel.promo_share)[0,1]:.3f}")
print(f"precio medio  sin promo = {panel[panel.promo_share < .01].price.mean():.2f}   "
      f"con promo = {panel[panel.promo_share > .5].price.mean():.2f}")

specs = fit_specifications(panel)
for name, m in specs.items():
    c = 'log_price' if 'log_price' in m.params else 'discount'
    print(f"\n[{name}]\n    {c} = {m.params[c]:+.3f}   p = {m.pvalues[c]:.3f}   R² = {m.rsquared:.3f}")

Antitranspirante 150 ml C   semanas=104   rango de precio observado=[46.36, 63.39]
corr(log precio, promo_share) = -0.934
precio medio  sin promo = 60.33   con promo = 49.58

[base: log_price + tendencia + mes]
    log_price = -2.978   p = 0.000   R² = 0.945

[control por "hay combo activo"]
    log_price = -2.580   p = 0.017   R² = 0.945

[solo combo y profundidad (sin precio)]
    discount = +1.794   p = 0.302   R² = 0.940


La elasticidad queda entre **-2.98 y -2.58** al agregar el control de combo activo. Reporto el rango porque los datos no permiten separar con precisión precio, visibilidad y promoción.

## Simulador

La demanda se calcula dentro del rango histórico de precios y se convierte en ingreso y margen usando el costo reciente del SKU. No extrapolo fuera de ese rango.

In [9]:
r = run_elasticity()
print(f"Elasticidad usada: {r['elasticity']:.2f}  (rango entre especificaciones: "
      f"{min(r['elasticity_range']):.2f} a {max(r['elasticity_range']):.2f})")
print(f"Ancla: precio_ref = {r['ref_price']:.2f}, demanda_ref = {r['ref_qty']:.0f} u/sem, "
      f"costo = {r['econ']['unit_cost']:.2f}")

# ejemplo de uso: consultar el simulador en 3 precios
r['sim']([r['price_min'], r['ref_price'], r['price_max']]).round(2)

Elasticidad usada: -2.98  (rango entre especificaciones: -2.98 a -2.58)
Ancla: precio_ref = 63.34, demanda_ref = 366 u/sem, costo = 46.85


,price,demanda_esperada,ingreso,margen_abs,margen_pct,fuera_de_rango_observado
0,46.36,927.33,42994.40,-447.31,-0.01,False
1,63.34,366.25,23196.55,6039.14,0.26,False
2,63.39,365.36,23159.00,6043.39,0.26,False


In [10]:
g = r['grid']
print(f"Precio que maximiza MARGEN $ (en rango observado): {g.loc[g.margen_abs.idxmax(),'price']:.2f}")
print(f"Precio que maximiza INGRESO $:                     {g.loc[g.ingreso.idxmax(),'price']:.2f}")
print(f"Costo unitario:                                    {r['econ']['unit_cost']:.2f}")
print(f"Descuento de equilibrio del SKU:                   {r['econ']['breakeven_discount']:.1%}")

# ¿el óptimo teórico sin restricción de rango es siquiera alcanzable?
p_star = r['econ']['unit_cost'] * r['elasticity'] / (r['elasticity'] + 1)
print(f"\nÓptimo teórico de margen sin restricción: {p_star:.2f}  ->  "
      f"{'DENTRO' if r['price_min'] <= p_star <= r['price_max'] else 'FUERA'} del rango observado "
      f"[{r['price_min']:.2f}, {r['price_max']:.2f}]  (referencia, no recomendación)")

Precio que maximiza MARGEN $ (en rango observado): 63.39
Precio que maximiza INGRESO $:                     46.36
Costo unitario:                                    46.85
Descuento de equilibrio del SKU:                   23.1%

Óptimo teórico de margen sin restricción: 70.53  ->  FUERA del rango observado [46.36, 63.39]  (referencia, no recomendación)


In [11]:
img = mpimg.imread(PROJECT_ROOT / "report/reto_b_simulador.png")
plt.figure(figsize=(10, 6)); plt.imshow(img); plt.axis('off'); plt.show()

### Lectura

El precio que maximiza ingreso ($46.36) queda por debajo del costo unitario ($46.85). Dentro del rango observado, el margen mejora al acercarse al precio de lista. No recomiendo profundizar el descuento de este SKU.

Si el objetivo es volumen, probaría una mecánica que conserve el precio unitario, por ejemplo un bundle entre SKUs o un regalo por compra.

El resultado sigue limitado por la falta de variación de precio fuera de promociones, la colinealidad con el calendario y el aumento del costo unitario. Con más datos haría una prueba de precios por bodega.

---
# Reto C | Uplift promocional

Para cada combo entrené un modelo estacional con semanas sin promoción del mismo SKU y lo usé para estimar la demanda que habría ocurrido durante la ventana.

Analicé los 19 combos para comparar resultados con la misma regla.

## Rentabilidad

margen incremental = I × (P - C) - A_promo × P × d

I son las unidades incrementales; A_promo, las unidades vendidas durante la promoción; P, el precio de lista; C, el costo unitario; y d, el descuento.

Reporto cobertura como uplift observado dividido entre uplift requerido. Una cobertura mayor a 1 indica que la promoción recuperó el costo del descuento.

In [12]:
from uplift import PROMOS, estimate_uplift, run_all, plot_promos

detalle = plot_promos(df)
detalle.T

,0,1
sku,Antitranspirante 150 ml C,Desodorante 150 ml A
promo,Combo Verano 2,Combo Quincena Desodorante
inicio,2026-03-02,2025-08-04
fin,2026-05-10,2025-08-31
semanas,10,4
descuento,0.161,0.201
descuento_equilibrio,0.231,0.18
vende_bajo_costo,False,True
unidades_promo,8122,7738
unidades_reales,8122,7738


In [13]:
img = mpimg.imread(PROJECT_ROOT / "report/reto_c_uplift.png")
plt.figure(figsize=(12, 9)); plt.imshow(img); plt.axis('off'); plt.show()

### Comprobación del margen

Calculé el margen de dos formas: con los montos facturados y con la descomposición anterior. Ambas deberían producir un resultado parecido.

In [14]:
allc = run_all(df)
allc['descomposicion'] = allc.ganancia_por_volumen - allc.costo_del_descuento
allc['discrepancia_%'] = ((allc.margen_incremental - allc.descomposicion)
                          / allc.costo_del_descuento * 100).round(2)
print("Discrepancia máxima entre la vía empírica y la analítica: "
      f"{allc['discrepancia_%'].abs().max():.2f}%")
allc[['promo', 'margen_incremental', 'descomposicion', 'discrepancia_%']].head(8)

Discrepancia máxima entre la vía empírica y la analítica: 1.47%


,promo,margen_incremental,descomposicion,discrepancia_%
0,Combo Verano 2,-24643,-24779,0.17
1,Combo Verano Antitranspirante,-18961,-19189,0.43
2,Combo Quincena,-21998,-21623,-0.73
3,Combo Verano Desodorante,-45183,-45300,0.12
4,Combo Verano Desodorante 2,-69889,-69978,0.06
5,Combo Cierre Trimestre,-23578,-23558,-0.04
6,Combo Cabello Sano,-12567,-12587,0.10
7,Combo Cabello 3,-9548,-9589,0.30


La diferencia entre los dos cálculos es menor a 1.5%. El chequeo también mostró que no podía usar un costo histórico fijo: el costo unitario sube entre 5% y 6% al año. Por eso precio y costo se calculan dentro de cada ventana promocional.

Durante un combo, todas las ventas del SKU aparecen como promocionales. El descuento afecta tanto a las unidades adicionales como a las que se habrían vendido de cualquier forma.

In [15]:
print("Participación promocional dentro de la ventana de cada combo:")
print(f"   mínimo = {allc.share_promo_en_ventana.min():.0%}   "
      f"máximo = {allc.share_promo_en_ventana.max():.0%}")
print("\n-> La canibalización es total: toda unidad que se habría vendido igual")
print("   se vende con descuento. Por eso el 'costo del descuento' se aplica sobre")
print("   el volumen completo y no sólo sobre las unidades incrementales.")

Participación promocional dentro de la ventana de cada combo:
   mínimo = 100%   máximo = 100%

-> La canibalización es total: toda unidad que se habría vendido igual
   se vende con descuento. Por eso el 'costo del descuento' se aplica sobre
   el volumen completo y no sólo sobre las unidades incrementales.


### Resultados de las 19 promociones

In [16]:
cols = ['sku', 'promo', 'descuento', 'descuento_equilibrio', 'vende_bajo_costo',
        'uplift_obs_pct', 'uplift_req_pct', 'cobertura', 'margen_incremental']
allc[cols]

,sku,promo,descuento,descuento_equilibrio,vende_bajo_costo,uplift_obs_pct,uplift_req_pct,cobertura,margen_incremental
0,Antitranspirante 150 ml C,Combo Verano 2,0.161,0.231,False,93.3,135.2,0.69,-24643
1,Antitranspirante 150 ml C,Combo Verano Antitranspirante,0.152,0.231,False,72.6,113.9,0.64,-18961
2,Antitranspirante 150 ml C,Combo Quincena,0.202,0.231,False,103.2,177.9,0.58,-21998
3,Desodorante 150 ml A,Combo Verano Desodorante,0.152,0.180,False,78.5,150.2,0.52,-45183
4,Desodorante 150 ml A,Combo Verano Desodorante 2,0.172,0.180,False,92.8,183.8,0.51,-69889
5,Antitranspirante 150 ml C,Combo Cierre Trimestre,0.220,0.231,False,87.8,179.3,0.49,-23578
6,Shampoo Rizos 135 ml,Combo Cabello Sano,0.152,0.213,False,38.8,99.1,0.39,-12567
7,Shampoo Rizos 135 ml,Combo Cabello 3,0.102,0.213,False,17.1,55.9,0.31,-9548
8,Shampoo Rizos 135 ml,Combo Regreso a Clases Cabello,0.122,0.213,False,21.0,69.2,0.30,-13355
9,Shampoo 135 ml Azul,Combo Azul Otoño,0.111,0.206,False,14.3,61.6,0.23,-5752


## Conclusiones del Reto C

Ninguna promoción alcanzó cobertura 1. La mejor fue Combo Verano 2, con 0.69.

**Probar de nuevo:** Combo Verano 2, con menor descuento y en un grupo limitado de bodegas. Fue la promoción más cercana al equilibrio, pero todavía perdió margen.

**No repetir:** Combo Quincena Desodorante y Combo Cierre Trimestre Desodorante. Sus descuentos superaron el límite de 18.0% del SKU y vendieron por debajo del costo.

Combo Temporada Fría sirve como contraste: descontó sólo 10%, pero perdió $61,816 porque el volumen casi no respondió. Un descuento bajo no garantiza rentabilidad.

---
# Archivos de entrega

- Documento de dos páginas: report/VEMIO_metodologia_hallazgos.docx
- Resultados intermedios: data/
- Código reutilizable: src/